# 04 — Sequence Distance Features

This version uses the actual patient-record structure from Notebook 01.

Each patient record contains a six-hour sequence under `record["sequence"]`. Each discriminative pattern from Notebook 03 currently represents one hourly itemset.

For every pattern, the notebook compares it with every hour in the patient's recent window, keeps the closest match, and converts that distance into a similarity feature.

Similarity = 1 − itemset distance.


In [ ]:
from pathlib import Path
import ast
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "outputs").exists() and (PROJECT_ROOT.parent / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SEQ_DIR = PROJECT_ROOT / "outputs" / "sequences"
PATTERN_DIR = PROJECT_ROOT / "outputs" / "patterns"
FEATURE_DIR = PROJECT_ROOT / "outputs" / "features"
FIGURE_DIR = FEATURE_DIR / "figures"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

POSITIVE_SEQ_FILE = SEQ_DIR / "positive_sequences.pkl"
NEGATIVE_SEQ_FILE = SEQ_DIR / "negative_sequences.pkl"
PATTERN_FILE = PATTERN_DIR / "discriminative_patterns.csv"

print("Project root:", PROJECT_ROOT)


## 1. Load inputs

In [ ]:
with open(POSITIVE_SEQ_FILE, "rb") as f:
    positive_records = pickle.load(f)

with open(NEGATIVE_SEQ_FILE, "rb") as f:
    negative_records = pickle.load(f)

patterns_df = pd.read_csv(PATTERN_FILE)

print("Positive windows:", len(positive_records))
print("Negative windows:", len(negative_records))
print("Discriminative patterns:", len(patterns_df))
display(patterns_df)


## 2. Parse the actual sequence and pattern structures

The patient files contain dictionaries. The actual hourly sequence is stored under the `sequence` key.

Notebook 03 stores the mined pattern in `pattern_text`, so that column is used here.


In [ ]:
def parse_pattern_itemset(value):
    text = str(value).strip()
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (tuple, list)):
            return set(str(x) for x in parsed)
        return {str(parsed)}
    except (ValueError, SyntaxError):
        return {text}

def extract_sequence(record):
    if isinstance(record, dict):
        sequence = record.get("sequence", [])
    else:
        sequence = record
    return [set(str(x) for x in hour) for hour in (sequence or [])]

def itemset_distance(a, b):
    a, b = set(a), set(b)
    union = a | b
    if not union:
        return 0.0
    return 1.0 - len(a & b) / len(union)

def itemset_similarity(a, b):
    return 1.0 - itemset_distance(a, b)

patterns_df["pattern_itemset"] = patterns_df["pattern_text"].apply(parse_pattern_itemset)
patterns_df["pattern_size"] = patterns_df["pattern_itemset"].apply(len)

display(patterns_df[[
    "pattern_text", "pattern_size", "positive_support",
    "negative_support", "support_difference", "support_ratio"
]])


## 3. Representation sanity check

The patient should contain multiple hourly itemsets, while each current discriminative pattern is one hourly itemset.


In [ ]:
example_record = positive_records[0]
example_sequence = extract_sequence(example_record)
example_pattern = patterns_df.iloc[0]["pattern_itemset"]

print("Patient:", example_record.get("patient_id", "unknown"))
print("Number of hours:", len(example_sequence))

for i, hour in enumerate(example_sequence, 1):
    print(f"Hour {i}:", sorted(hour))

print("\nFirst pattern:", sorted(example_pattern))

print("\nHourly distances to first pattern:")
for i, hour in enumerate(example_sequence, 1):
    d = itemset_distance(hour, example_pattern)
    print(f"Hour {i}: distance={d:.4f}, similarity={1-d:.4f}")


## 4. Generate distance-based features

For every patient and every discriminative pattern:
1. compare the pattern with every hour;
2. select the minimum distance;
3. convert it to similarity.

This produces one similarity feature per mined pattern.


In [ ]:
def build_features(records, patterns):
    rows = []
    pattern_sets = patterns["pattern_itemset"].tolist()

    for record in records:
        hours = extract_sequence(record)

        row = {
            "patient_id": record.get("patient_id", "unknown")
            if isinstance(record, dict) else "unknown",
            "sequence_length": len(hours)
        }

        similarities = []

        for j, pattern in enumerate(pattern_sets, 1):
            if hours:
                distances = [itemset_distance(hour, pattern) for hour in hours]
                best_distance = min(distances)
                best_hour = int(np.argmin(distances) + 1)
            else:
                best_distance = 1.0
                best_hour = -1

            similarity = 1.0 - best_distance
            similarities.append(similarity)

            row[f"pattern_{j:02d}_distance"] = best_distance
            row[f"pattern_{j:02d}_similarity"] = similarity
            row[f"pattern_{j:02d}_best_hour"] = best_hour

        row["max_pattern_similarity"] = max(similarities) if similarities else 0.0
        row["mean_pattern_similarity"] = float(np.mean(similarities)) if similarities else 0.0
        rows.append(row)

    return pd.DataFrame(rows)

positive_features = build_features(positive_records, patterns_df)
negative_features = build_features(negative_records, patterns_df)

positive_features["label"] = 1
negative_features["label"] = 0

features_df = pd.concat([positive_features, negative_features], ignore_index=True)

print("Feature matrix shape:", features_df.shape)
display(features_df.head())


## 5. Similarity diagnostics

In [ ]:
similarity_columns = [
    c for c in features_df.columns
    if c.startswith("pattern_") and c.endswith("_similarity")
]

summary_rows = []
for c in similarity_columns:
    summary_rows.append({
        "feature": c,
        "positive_mean": positive_features[c].mean(),
        "negative_mean": negative_features[c].mean(),
        "mean_difference": positive_features[c].mean() - negative_features[c].mean(),
        "positive_median": positive_features[c].median(),
        "negative_median": negative_features[c].median()
    })

similarity_summary = pd.DataFrame(summary_rows).sort_values(
    "mean_difference", ascending=False
)

display(similarity_summary)


In [ ]:
comparison = pd.DataFrame({
    "group": ["Positive", "Negative"],
    "max_similarity": [
        positive_features["max_pattern_similarity"].mean(),
        negative_features["max_pattern_similarity"].mean()
    ],
    "mean_similarity": [
        positive_features["mean_pattern_similarity"].mean(),
        negative_features["mean_pattern_similarity"].mean()
    ]
})

display(comparison)

x = np.arange(2)
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, comparison["max_similarity"], width, label="Max similarity")
plt.bar(x + width/2, comparison["mean_similarity"], width, label="Mean similarity")
plt.xticks(x, comparison["group"])
plt.ylabel("Average similarity")
plt.title("Pattern similarity by cohort")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "cohort_similarity_comparison.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
for c in similarity_columns:
    plt.figure(figsize=(7, 4))
    plt.hist(positive_features[c], bins=20, alpha=0.6, label="Positive")
    plt.hist(negative_features[c], bins=20, alpha=0.6, label="Negative")
    plt.xlabel("Best hourly similarity")
    plt.ylabel("Frequency")
    plt.title(f"Similarity distribution: {c}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / f"{c}_distribution.png", dpi=200, bbox_inches="tight")
    plt.show()


## 6. Build and save the classification matrix

Notebook 05 will use the similarity features and the two aggregate similarity features. Distance and best-hour columns are retained in the complete CSV for interpretation.


In [ ]:
feature_columns = [
    c for c in features_df.columns
    if c.endswith("_similarity")
]

X = features_df[feature_columns].copy()
y = features_df["label"].astype(int)

FEATURE_FILE = FEATURE_DIR / "sequence_distance_features.csv"
SUMMARY_FILE = FEATURE_DIR / "similarity_feature_summary.csv"
METADATA_FILE = FEATURE_DIR / "feature_metadata.csv"

features_df.to_csv(FEATURE_FILE, index=False)
similarity_summary.to_csv(SUMMARY_FILE, index=False)

metadata = []
for i, row in patterns_df.reset_index(drop=True).iterrows():
    n = i + 1
    metadata.append({
        "feature": f"pattern_{n:02d}_similarity",
        "pattern_text": row["pattern_text"],
        "positive_support": row.get("positive_support", np.nan),
        "negative_support": row.get("negative_support", np.nan),
        "support_difference": row.get("support_difference", np.nan),
        "support_ratio": row.get("support_ratio", np.nan)
    })

metadata += [
    {
        "feature": "max_pattern_similarity",
        "pattern_text": "Maximum similarity across discriminative patterns"
    },
    {
        "feature": "mean_pattern_similarity",
        "pattern_text": "Mean similarity across discriminative patterns"
    }
]

pd.DataFrame(metadata).to_csv(METADATA_FILE, index=False)

print("Classification matrix:", X.shape)
print("Class counts:")
print(y.value_counts().sort_index())
print("\nSaved:", FEATURE_FILE)
print("Saved:", SUMMARY_FILE)
print("Saved:", METADATA_FILE)


## 7. Final sanity checks

In [ ]:
assert len(patterns_df) > 0
assert len(features_df) > 0
assert len(feature_columns) > 0
assert X.isna().sum().sum() == 0
assert ((X >= 0) & (X <= 1)).all().all()
assert set(y.unique()) == {0, 1}

print("All sanity checks passed.")
print(f"Ready for Notebook 05 with {len(feature_columns)} similarity features.")
